# Clasificación de imágenes MRI con CNN y LSTM
Experimento académico para distinguir imágenes ya segmentadas de imágenes sin segmentar. El modelo clasifica imágenes y la evaluación se refiere a las etiquetas del conjunto de datos.

## 1. Preparar los datos
En el panel **Archivos** de Colab, coloca tu conjunto de datos en `/content/datos` o cambia esa ruta. Conserva los nombres exactos de `FOLDERS` o ajusta el diccionario a tus carpetas. Confirma cuál grupo corresponde a cada clase con tu fuente original. Usa imágenes con permiso de uso y sin datos identificables.

La división incluida es estratificada por imagen. Si hay cortes del mismo paciente o pares segmentados y sin segmentar de un mismo estudio, asigna el grupo completo a un único conjunto antes de interpretar métricas. Esa división requiere identificadores de paciente o estudio.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import train_test_split

IMAGE_SIZE = (128, 128)
SEED = 42
FOLDERS = {
    "giloma_1": 0,
    "meningioma_1": 0,
    "tumor_1": 0,
    "giloma_2": 1,
    "meningoma_2": 1,
    "tumor_2": 1,
}
EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif"}

DATA_DIR = Path("/content/datos")
OUTPUT_DIR = Path("/content/resultados_mri")
EPOCHS = 30
tf.keras.utils.set_random_seed(SEED)


## 2. Cargar y separar las imágenes
OpenCV convierte las imágenes a escala de grises y las redimensiona a 128 × 128. Se normalizan a valores de 0 a 1. La separación por imagen reserva aproximadamente 60 % para entrenamiento, 20 % para validación y 20 % para prueba.

In [ ]:
def load_images(data_dir):
    images, labels = [], []
    for name, label in FOLDERS.items():
        folder = data_dir / name
        if not folder.is_dir():
            raise ValueError(f"No se encontró la carpeta {folder}")

        count = 0
        for path in sorted(folder.iterdir()):
            if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
                continue
            encoded = np.fromfile(path, dtype=np.uint8)
            image = None
            if encoded.size:
                image = cv2.imdecode(encoded, cv2.IMREAD_GRAYSCALE)
            if image is None:
                print(f"Imagen ilegible omitida: {path}")
                continue
            image = cv2.resize(image, IMAGE_SIZE)
            images.append(image.astype(np.float32) / 255.0)
            labels.append(label)
            count += 1
        if count == 0:
            raise ValueError(f"No hay imágenes legibles en {folder}")
        print(f"{name}: {count} imágenes, clase {label}")

    return np.asarray(images)[..., np.newaxis], np.asarray(labels)

def split_data(images, labels):
    try:
        x_development, x_test, y_development, y_test = train_test_split(
            images, labels, test_size=0.20, stratify=labels, random_state=SEED
        )
        x_train, x_val, y_train, y_val = train_test_split(
            x_development, y_development, test_size=0.25,
            stratify=y_development, random_state=SEED
        )
    except ValueError as error:
        raise ValueError(f"Datos insuficientes para la división estratificada: {error}") from error

    splits = ((x_train, y_train), (x_val, y_val), (x_test, y_test))
    if any(len(np.unique(split_labels)) != 2 for _, split_labels in splits):
        raise ValueError("Cada conjunto debe contener ambas clases.")
    return splits

images, labels = load_images(DATA_DIR)
(x_train, y_train), (x_val, y_val), (x_test, y_test) = split_data(images, labels)
for name, split_labels in [("Entrenamiento", y_train), ("Validación", y_val), ("Prueba", y_test)]:
    print(name, len(split_labels), "| ejemplos por clase:", np.bincount(split_labels))


## 3. Construir la red
La CNN extrae características visuales. La LSTM bidireccional recorre las filas del último mapa de características y la capa sigmoide produce un puntaje para la clase 1.

In [ ]:
def build_model():
    layers = tf.keras.layers
    inputs = layers.Input(shape=(*IMAGE_SIZE, 1))
    features = inputs
    for filters in (32, 64, 128):
        features = layers.Conv2D(filters, 3, activation="relu", padding="same")(features)
        features = layers.MaxPooling2D(2)(features)
    features = layers.Conv2D(256, 3, activation="relu", padding="same")(features)

    height, width, channels = features.shape[1:]
    sequence = layers.Reshape((height, width * channels))(features)
    encoded = layers.Bidirectional(layers.LSTM(64))(sequence)
    encoded = layers.Dense(128, activation="relu")(encoded)
    encoded = layers.Dropout(0.4)(encoded)
    outputs = layers.Dense(1, activation="sigmoid")(encoded)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model()
model.summary()

## 4. Entrenar
Se aplican transformaciones únicamente al conjunto de entrenamiento. El modelo se guarda cuando mejora la pérdida de validación. Para comprobar el flujo puedes cambiar `EPOCHS` a 1 en la primera celda.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = OUTPUT_DIR / "modelo.keras"
augmentation = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15, width_shift_range=0.06, height_shift_range=0.06,
    zoom_range=0.08, horizontal_flip=True,
)
batches = augmentation.flow(x_train, y_train, batch_size=16, seed=SEED)
model = build_model()
history = model.fit(
    batches, validation_data=(x_val, y_val), epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(str(best_model_path), monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6),
    ],
)


## 5. Evaluar en prueba
Recarga el mejor modelo y usa el conjunto reservado. Se muestran el reporte por clase, la matriz de confusión y la curva ROC. Los archivos se guardan en `/content/resultados_mri` y se pueden descargar desde el panel **Archivos**.

In [ ]:
def save_plots(history, labels, predictions, scores, output_dir):
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    for axis, metric, title in zip(
        axes, ("loss", "accuracy"), ("Pérdida por época", "Exactitud por época")
    ):
        axis.plot(history[metric], label="Entrenamiento")
        axis.plot(history[f"val_{metric}"], label="Validación")
        axis.set_title(title)
        axis.set_xlabel("Índice de época")
        axis.legend()
    figure.tight_layout()
    figure.savefig(output_dir / "entrenamiento.png", dpi=150)
    plt.close(figure)

    confusion = ConfusionMatrixDisplay.from_predictions(labels, predictions, labels=[0, 1])
    confusion.ax_.set(xlabel="Clase predicha", ylabel="Clase real")
    confusion.figure_.savefig(output_dir / "matriz_confusion.png", dpi=150)
    plt.close(confusion.figure_)

    roc = RocCurveDisplay.from_predictions(labels, scores)
    roc.ax_.set(xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos")
    roc.figure_.savefig(output_dir / "curva_roc.png", dpi=150)
    plt.close(roc.figure_)

best_model = tf.keras.models.load_model(best_model_path)
scores = best_model.predict(x_test, verbose=0).ravel()
predictions = (scores >= 0.5).astype(int)
report = classification_report(y_test, predictions, labels=[0, 1], zero_division=0)
print(report)
(OUTPUT_DIR / "reporte_prueba.txt").write_text(report, encoding="utf-8")
save_plots(history.history, y_test, predictions, scores, OUTPUT_DIR)
from IPython.display import Image, display

for filename in ("entrenamiento.png", "matriz_confusion.png", "curva_roc.png"):
    display(Image(filename=str(OUTPUT_DIR / filename)))
